# 의안 메타데이터 LoRA 학습 (Colab / Kaggle)

- **데이터:** `finetune_dataset.jsonl` (`instruction` / `input` / `output`)
- **스크립트:** 같은 저장소의 `finetune/train_lora.py`
- 순서: Colab → `README-colab.md` / Kaggle → `README-kaggle.md` / 공통 → `README-colab-kaggle.md`

**Colab:** 런타임 → **L4** 우선(없으면 T4 + `--use-4bit`).

**Kaggle:** Accelerator GPU, 데이터 경로를 `/kaggle/input/...`로 바꿔 실행.


In [ ]:
# Colab: finetune_dataset.jsonl 을 /content/ 에 업로드한 뒤 경로 확인
# Kaggle: Add Data 로 Dataset 연결 후 아래 경로 수정
#
# train_lora.py 가 들어 있는 폴더 (저장소를 !git clone 했다면 .../metadata/finetune 등)
FINETUNE_DIR = "/content"

DATA_PATH = "/content/finetune_dataset.jsonl"  # 예: "/kaggle/input/my-dataset/finetune_dataset.jsonl"
OUTPUT_DIR = "/content/lora-output"  # 예: "/kaggle/working/lora-output"
BASE_MODEL = (
    "Bllossom/llama-3.2-Korean-Bllossom-3B"  # "MLP-KTLim/llama-3-Korean-Bllossom-8B"
)

In [ ]:
# 저장소 전체를 올렸다면: !pip install -q -r /content/metadata/finetune/requirements-train.txt
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes safetensors

In [ ]:
# (선택) 허브 게이트 모델·비공개 리소스: Colab Secrets 에 HF_TOKEN 후 아래 주석 해제
# import os
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from collections import deque
from pathlib import Path
import subprocess
import sys

script = Path(FINETUNE_DIR).resolve() / "train_lora.py"
if not script.is_file():
    raise FileNotFoundError(
        f"train_lora.py 없음: {script}\n"
        "FINETUNE_DIR 를 스크립트가 있는 폴더로 맞추세요 (예: 클론 경로의 .../finetune)."
    )

cmd = [
    sys.executable,
    "-u",  # 버퍼 끄기 (Colab에서 로그·Traceback 바로 보이게)
    str(script),
    "--data",
    DATA_PATH,
    "--out",
    OUTPUT_DIR,
    "--base-model",
    BASE_MODEL,
    "--epochs",
    "2",
    "--batch-size",
    "2",
    "--grad-accum",
    "4",
    "--max-length",
    "2048",
    "--use-4bit",
]

# 실시간 출력 + 실패 시 마지막 줄을 예외에 포함 (위쪽 스크롤 없이 원인 확인)
tail = deque(maxlen=500)
proc = subprocess.Popen(
    cmd,
    cwd=str(script.parent),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    tail.append(line)
    print(line, end="")
rc = proc.wait()
if rc != 0:
    msg = "".join(tail)
    raise RuntimeError(
        f"train_lora.py exit {rc}.\n\n"
        f"--- 마지막 출력(최대 {tail.maxlen}줄) ---\n{msg}"
    )

학습이 끝나면 `OUTPUT_DIR`을 zip으로 묶어 Drive에 저장하거나, Kaggle은 Output으로 다운로드한다.
